# Analisi GNSS con la pipeline condivisa

Eseguire prima il preprocessing e il training offline descritti nel README.
Questo notebook usa gli stessi indicatori e le stesse Random Forest del servizio di classificazione.
Le label restano disponibili per la valutazione, ma non entrano nell’inferenza.
Il solo `final_risk` è la stima della RF binaria.


In [ ]:
from pathlib import Path
import sys
import joblib
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from shared.preprocessing import RiskProcessor
from shared.inference import predict

CSV_PATH = ROOT / 'gnss_tracks/track_000_mixed.csv'
GEOMETRY_PATH = CSV_PATH.with_suffix('.geometry.csv')
processor = RiskProcessor.load(ROOT / 'models/preprocessing.joblib')
models = joblib.load(ROOT / 'models/random_forest_models.joblib')


In [ ]:
track = pd.read_csv(CSV_PATH)
geometry = pd.read_csv(GEOMETRY_PATH)
indicators = processor.transform(track, geometry)
predictions = predict(models, indicators)
result = indicators.join(predictions.drop(columns='timestamp'))
display(result)
output = ROOT / 'notebook_results' / f'{CSV_PATH.stem}_classified.csv'
output.parent.mkdir(exist_ok=True)
result.to_csv(output, index=False)
print(output)
